# Eksplorasi: Apakah `late_delivery` layak dilanjutkan ke pipeline penuh?

Tujuan notebook ini: cek dulu apakah ada sinyal statistik nyata di fitur terhadap target `is_late`, **sebelum** invest waktu bikin DAG + Optuna + DooD lengkap seperti yang sudah dilakukan untuk `customer_churn`.

Struktur:
1. Load parquet & cek struktur dasar
2. Distribusi target (`is_late`)
3. Feature engineering: turunkan jarak pengiriman dari koordinat lat/long
4. Diagnostik Mutual Information
5. Baseline model cepat (tanpa tuning) -- estimasi kasar PR-AUC
6. Kesimpulan

In [1]:
import pandas as pd
import numpy as np
import glob
import os

from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import mutual_info_score
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)

## 1. Load parquet & cek struktur dasar

Sesuaikan `PARQUET_DIR` kalau lokasinya beda. Sel di bawah akan list semua file `.parquet` yang ada supaya kamu bisa pastikan nama file yang benar (mengikuti nama model dbt-nya) sebelum di-load.

In [2]:
# Gunakan huruf 'r' di depan tanda kutip agar Windows membaca path dengan benar
PARQUET_DIR = r"D:\PREP_INTERN\PROJECT_DATAFLOW_SIMULATION\datasets\parquet"

print("File parquet yang tersedia:")
for f in sorted(glob.glob(os.path.join(PARQUET_DIR, "*.parquet"))):
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):<55s} ({size_kb:.1f} KB)")

File parquet yang tersedia:
  ml_churn_training_dataset_latest.parquet                (844.9 KB)
  ml_customer_churn_latest.parquet                        (988.8 KB)
  ml_customer_clv_latest.parquet                          (390.5 KB)
  ml_late_delivery_inference_latest.parquet               (672.5 KB)
  ml_late_delivery_training_latest.parquet                (401.7 KB)
  ml_product_return_latest.parquet                        (491.6 KB)
  ml_session_conversion_latest.parquet                    (18272.5 KB)


In [3]:
# GANTI nama file ini sesuai hasil list di atas -- kemungkinan mengikuti
# nama model dbt (misal: late_delivery_prediction_dataset_latest.parquet)
PARQUET_FILE = os.path.join(PARQUET_DIR, "ml_late_delivery_training_latest.parquet")

df = pd.read_parquet(PARQUET_FILE)

print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
df.head()

Shape: (15941, 15)

Dtypes:
order_id                          int64
is_late                           int64
order_day_of_week               float64
order_hour_of_day               float64
order_month                     float64
total_items                       int64
unique_categories_in_package      int64
total_order_value               float64
origin_warehouse                    str
destination_country                 str
destination_state                   str
dc_latitude                     float64
dc_longitude                    float64
user_latitude                   float64
user_longitude                  float64
dtype: object


,order_id,is_late,order_day_of_week,order_hour_of_day,order_month,total_items,unique_categories_in_package,total_order_value,origin_warehouse,destination_country,destination_state,dc_latitude,dc_longitude,user_latitude,user_longitude
0,60988,0,1.0,17.0,1.0,1,1,31.99,Memphis TN,United Kingdom,England,35.1174,-89.9711,53.436566,-2.193451
1,14529,0,6.0,5.0,1.0,1,1,28.00,Memphis TN,China,Liaoning,35.1174,-89.9711,41.934002,121.608338
2,21100,0,6.0,15.0,1.0,1,1,4.50,Port Authority of New York/New Jersey NY/NJ,Brasil,Ceará,40.6340,-73.7834,-4.109353,-38.466053
3,93742,0,3.0,3.0,1.0,1,1,99.65,Chicago IL,Spain,Andalucía,41.8369,-87.6847,37.262523,-6.944595
4,96913,1,3.0,8.0,1.0,3,2,236.88,Savannah GA,Germany,Sachsen-Anhalt,32.0167,-81.1167,51.345747,11.974885


In [4]:
print("Missing values per kolom:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Tidak ada missing value.")

Missing values per kolom:
Tidak ada missing value.


## 2. Distribusi target (`is_late`)

Cek dulu tingkat imbalance-nya -- sama seperti churn, ini menentukan apakah butuh `scale_pos_weight` dan berapa `n_splits` CV yang aman.

In [5]:
target_col = "is_late"

counts = df[target_col].value_counts()
pct = df[target_col].value_counts(normalize=True) * 100

print("Distribusi target:")
for cls in counts.index:
    print(f"  Kelas {cls}: {counts[cls]:>6d} ({pct[cls]:.2f}%)")

ratio = counts.min() / counts.max()
print(f"\nRasio minoritas:mayoritas = 1:{1/ratio:.1f}")

Distribusi target:
  Kelas 0:   9611 (60.29%)
  Kelas 1:   6330 (39.71%)

Rasio minoritas:mayoritas = 1:1.5


## 3. Feature Engineering: Jarak Pengiriman

Ini fitur turunan yang **secara fisik masuk akal** berhubungan dengan keterlambatan -- beda dengan kasus churn kemarin di mana fitur-fiturnya tidak punya hubungan kausal jelas ke label. Kalau data sintetis ini di-generate dengan mempertimbangkan jarak riil (bukan acak total), fitur ini punya peluang MI tinggi.

In [6]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Jarak great-circle antara dua titik koordinat, dalam km."""
    R = 6371.0  # radius bumi (km)
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

coord_cols = ['dc_latitude', 'dc_longitude', 'user_latitude', 'user_longitude']
if all(c in df.columns for c in coord_cols):
    df['shipping_distance_km'] = haversine_distance(
        df['dc_latitude'], df['dc_longitude'],
        df['user_latitude'], df['user_longitude']
    )
    print("Fitur 'shipping_distance_km' berhasil dibuat.")
    print(df['shipping_distance_km'].describe())
else:
    missing_coords = [c for c in coord_cols if c not in df.columns]
    print(f"Kolom koordinat tidak lengkap, tidak bisa hitung jarak. Hilang: {missing_coords}")

Fitur 'shipping_distance_km' berhasil dibuat.
count    15941.000000
mean      8113.416445
std       4187.042756
min          3.691278
25%       5662.141932
50%       7980.701120
75%      11773.246578
max      18720.309550
Name: shipping_distance_km, dtype: float64


## 4. Diagnostik Mutual Information

Sama seperti yang dipakai di `model_trainer.py` churn -- kalau MI mendekati 0 di semua fitur, itu tanda data ini juga belum punya sinyal yang bisa dipelajari.

In [7]:
# 1. Exclude kolom yang tidak diinginkan
exclude_cols = ['order_id', target_col, 'dc_latitude', 'dc_longitude', 'user_latitude', 'user_longitude']
feature_cols = [c for c in df.columns if c not in exclude_cols]

# 2. CARA AMAN: Ambil yang PASTI ANGKA dulu
numeric_features = df[feature_cols].select_dtypes(include=['number']).columns.tolist()

# 3. Sisanya (object, string, category, bool) otomatis masuk ke categorical
categorical_features = [c for c in feature_cols if c not in numeric_features]

y = df[target_col]

# ... Lanjutkan ke kode print dan Mutual Information Anda di bawahnya ...

print("=" * 80)
print("MUTUAL INFORMATION vs is_late")
print("=" * 80)

mi_results = []

if numeric_features:
    X_num = df[numeric_features].fillna(0)
    mi_numeric = mutual_info_classif(X_num, y, random_state=42)
    for feat, score in zip(numeric_features, mi_numeric):
        mi_results.append((feat, 'numeric', score))

if categorical_features:
    for feat in categorical_features:
        le = LabelEncoder()
        encoded = le.fit_transform(df[feat].astype(str).fillna("MISSING"))
        score = mutual_info_score(encoded, y)
        mi_results.append((feat, 'categorical', score))

mi_df = pd.DataFrame(mi_results, columns=['feature', 'type', 'mutual_information'])
mi_df = mi_df.sort_values('mutual_information', ascending=False).reset_index(drop=True)

for _, row in mi_df.iterrows():
    print(f"  [{row['type']:<11s}] {row['feature']:<35s} MI = {row['mutual_information']:.5f}")

print("=" * 80)
print("Catatan: MI < 0.01 di HAMPIR SEMUA fitur = kemungkinan besar tidak ada")
print("sinyal yang bisa dipelajari (seperti kasus customer_churn kemarin).")
print("MI beberapa fitur jelas lebih tinggi dari yang lain (>0.02-0.05) =")
print("kandidat kuat untuk dilanjutkan ke pipeline penuh.")
print("=" * 80)

MUTUAL INFORMATION vs is_late
  [numeric    ] order_day_of_week                   MI = 0.00956
  [categorical] destination_state                   MI = 0.00689
  [numeric    ] unique_categories_in_package        MI = 0.00457
  [numeric    ] shipping_distance_km                MI = 0.00288
  [categorical] destination_country                 MI = 0.00055
  [categorical] origin_warehouse                    MI = 0.00028
  [numeric    ] order_hour_of_day                   MI = 0.00000
  [numeric    ] total_order_value                   MI = 0.00000
  [numeric    ] order_month                         MI = 0.00000
  [numeric    ] total_items                         MI = 0.00000
Catatan: MI < 0.01 di HAMPIR SEMUA fitur = kemungkinan besar tidak ada
sinyal yang bisa dipelajari (seperti kasus customer_churn kemarin).
MI beberapa fitur jelas lebih tinggi dari yang lain (>0.02-0.05) =
kandidat kuat untuk dilanjutkan ke pipeline penuh.


## 5. Baseline Model Cepat (tanpa tuning)

Estimasi kasar PR-AUC pakai XGBoost default + `scale_pos_weight` sederhana -- cuma untuk gambaran cepat, BUKAN model final. Kalau PR-AUC ini jauh di atas baseline (proporsi kelas minoritas), berarti ada sinyal nyata yang layak dioptimalkan lebih lanjut.

In [8]:
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from category_encoders import TargetEncoder
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

X = df[feature_cols]

numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=0))])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_encoder', TargetEncoder())
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
], remainder='drop')

count_minority = y.value_counts().min()
count_majority = y.value_counts().max()
scale_pos_weight = count_minority / count_majority if y.value_counts().idxmax() == 1 else count_majority / count_minority
n_splits = min(5, max(2, count_minority))

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        objective='binary:logistic', tree_method='hist',
        random_state=42, n_jobs=2, eval_metric='logloss'
    ))
])

oof_proba = cross_val_predict(
    pipeline, X, y,
    cv=StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42),
    method='predict_proba', n_jobs=1
)[:, 1]

pr_auc = average_precision_score(y, oof_proba)
baseline_pr_auc = y.mean()  # PR-AUC baseline = proporsi kelas positif

print(f"PR-AUC model (out-of-fold, tanpa tuning): {pr_auc:.4f}")
print(f"PR-AUC baseline (tebak proporsi kelas)  : {baseline_pr_auc:.4f}")
print(f"Lift terhadap baseline                  : {pr_auc / baseline_pr_auc:.2f}x")

PR-AUC model (out-of-fold, tanpa tuning): 0.3977
PR-AUC baseline (tebak proporsi kelas)  : 0.3971
Lift terhadap baseline                  : 1.00x


## 6. Kesimpulan

Cek dua angka di atas:

- **MI hampir semua < 0.01 DAN lift PR-AUC mendekati 1.0x** -> pola sama seperti `customer_churn`, kemungkinan besar data ini juga belum punya sinyal. Tidak perlu lanjut ke pipeline penuh.
- **Ada beberapa fitur MI jelas lebih tinggi (>0.02) DAN lift PR-AUC > 1.5x-2x** -> ada sinyal nyata, layak dilanjutkan ke DAG training penuh (Optuna + DooD) seperti pola `customer_churn`.

Isi kesimpulan manual di sini setelah lihat hasil di atas: ___________